In [1]:
import pandas as pd

# Procedure
'procedure_source_concept_id' contains IDs, which we want to match exactly to any of the IDs in 'ids_df'

When 'procedure_source_concept_id' is 0, we are also interested in exact matches from 'procedure_source_value' to codes in 'codes_long_df', for entries where 'codes_long_df' starts with a code and in codes containing 'measle' or 'mmr' but not containing 'allerg' or 'reaction'

In [ ]:
ids_df = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_id.csv')  
codes_df = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_code.csv') 
codes_long_df = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_long_code.csv')  

# Add regex pattern for "starts with code followed by |"
regex_codes_pipe = '|'.join([f'^{code}\\|' for code in code_list])
regex_codes_pipe = regex_codes_pipe.replace('\\', '\\\\')

id_list = ids_df['concept_id'].dropna().astype(int).unique().tolist()
code_list = codes_long_df['concept_code'].dropna().astype(str).unique().tolist()

# format ID list for SQL IN clause
ids_sql = ', '.join(map(str, id_list))

# format code list for SQL IN clause
codes_sql = ', '.join(f"'{code}'" for code in code_list)

# regex pattern for keyword search (case-insensitive)
regex_include = '(?i)(measle|mmr)'
regex_exclude = '(?i)(allerg|reaction)'


client = bq.Client(project='law-nero-phi-dho-scc-covid')

PROJECT_ID = "law-nero-phi-dho-scc-covid"
DATASET = "afc0125"
TABLE = "Drug"

for i in range(1990,2026):
                
    QUERY = f"""

    SELECT *
    FROM {PROJECT_ID}.{DATASET}.{TABLE} WHERE 
    EXTRACT(YEAR FROM encounterdate) = {i} AND
    procedure_concept_id IN ({ids_sql}) OR
    (
        procedure_concept_id = 0 AND (
            procedure_source_value IN ({codes_sql}) OR
            REGEXP_CONTAINS(procedure_source_value, '{regex_codes_pipe}') OR
            (REGEXP_CONTAINS(procedure_source_value, '{regex_include}')
             AND NOT REGEXP_CONTAINS(procedure_source_value, '{regex_exclude}'))
        )
    )

    """

    query_job = client.query(QUERY)
    df = query_job.to_dataframe()

    df.to_csv(f"/share/pi/deho-pi/AFC/BQ/Drug_0125_measles/Drug_0125_{i}.csv.gz", compression = "gzip", escapechar='\\')

    del df
    del query_job
